In [2]:
import pandas as pd
import dask.dataframe as df
import sqlalchemy as sq
import requests
import kagglehub
import psycopg2
import numpy as np
import json
import glob
import os
import polars as pl
import shutil
import re
import zipfile




In [2]:
# #Define file path for source data csv files

# file_paths = glob.glob('SourceData/*.csv')

# output_folder = 'CleanedData'


# # Define function to perform auto cleaning on files

# def clean_source_files(df):
#     df = df.drop(columns=['_DENSTR2', 'PRECALL', 'REPNUM', 'REPDEPTH', 'HAVHPAD','_GEOSTR','FMONTH', 'DISPCODE','CTELENUM', 'NUMADULT', 'NUMWOMEN', 'CTELNUM1', 'CELLFON2',
#         'IYEAR', 'NUMMEN'], axis = 1, errors='ignore')
#     df.fillna(value=9.0, inplace=True)
#     return df

# # Loop through files 

# for file in file_paths:
#     print(f'processing: {file}')

#     # Reading and cleaning files 
#     df = pd.read_csv(file)
#     cleaned_df = clean_source_files(df)

#     # Get the file name from the full file path

#     file_name = os.path.basename(file)

#     # Define the save path for the new files 

#     save_path = os.path.join(output_folder, file_name)

#     # Save cleaned file

#     cleaned_df.to_csv(save_path, index=False)
#     print(f"Saved: {save_path}")





In [ ]:
# Above process rewritten using Polars as the above took almost 8 minutes

def Remove_Unneed_Columns(df):
    return(
    df.drop(['_DENSTR2', 'PRECALL', 'REPNUM', 'REPDEPTH', 'HAVHPAD','_GEOSTR','FMONTH','CTELENUM', 'NUMADULT', 'NUMWOMEN', 'CTELNUM1', 'CELLFON2',
        'IYEAR', 'NUMMEN', 'INTVID', 'SEQNO', '_PSU', 'NATTMPTS', 'CELLFON'], strict=False)
    .fill_null(9.0)
    )

def Set_Date_Values(df):
    return(
        df.with_columns([pl.col('IDATE').str.replace_all(r"b'", '').str.replace_all('"', '').str.replace_all('0931', '0930').str.replace_all(' ', '0').str.replace_all("'", '')]
    ))


#Define output folder, deleting and recreating if already exists

output_folder = 'CleanedData'
if os.path.exists(output_folder):
    shutil.rmtree(output_folder) # deletes output folder if exists

# Recreate output folder 

os.makedirs(output_folder, exist_ok=True)

# Unzip source files

zip_file_path = 'SourceData/Archive.zip'
destination_directory = 'SourceData'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(destination_directory)

print(f'All files successfully unzipped to {destination_directory}')


#Process Files

files = glob.glob('SourceData/*.csv')

for file in files:
    print('Processing {file}')
    df = pl.read_csv(file)
    df_cleaned = Remove_Unneed_Columns(df)

    file_name = os.path.basename(file)
    save_path = os.path.join(output_folder, file_name)
    df_cleaned.write_csv(save_path)

    print(f'Processed {file_name} with Polars in super fast time')
    

All files successfully unzipped to SourceData
Processing {file}
Processed 2015.csv with Polars in super fast time
Processing {file}
Processed 2014.csv with Polars in super fast time
Processing {file}
Processed 2013.csv with Polars in super fast time
Processing {file}
Processed 2012.csv with Polars in super fast time
Processing {file}
Processed 2011.csv with Polars in super fast time


In [5]:
# Turing the IDATE column into an actual date
Cleaned_2011 = pl.read_csv('CleanedData/2011.csv')

print(Cleaned_2011[:, :5])



Cleaned_2011_Year_Clean = Cleaned_2011.with_columns([
    pl.col('IDATE').str.replace_all(r"b'", '').str.replace_all('"', '').str.replace_all('0931', '0930').str.replace_all(' ', '0').str.replace_all("'", '')
])

W_Year_2011 = Cleaned_2011_Year_Clean.with_columns(    # This found that 63 records had a date of 09/31 which is impossible so those dates will be updated to /09/30
    pl.col('IDATE').str.to_date("%m%d%Y")
)



#W_Year_2011

shape: (506_467, 5)
┌────────┬─────────────┬────────┬───────┬──────────┐
│ _STATE ┆ IDATE       ┆ IMONTH ┆ IDAY  ┆ DISPCODE │
│ ---    ┆ ---         ┆ ---    ┆ ---   ┆ ---      │
│ f64    ┆ str         ┆ str    ┆ str   ┆ f64      │
╞════════╪═════════════╪════════╪═══════╪══════════╡
│ 1.0    ┆ b'01202011' ┆ b'01'  ┆ b'20' ┆ 120.0    │
│ 1.0    ┆ b'01142011' ┆ b'01'  ┆ b'14' ┆ 120.0    │
│ 1.0    ┆ b'01062011' ┆ b'01'  ┆ b'06' ┆ 120.0    │
│ 1.0    ┆ b'02012011' ┆ b'02'  ┆ b'01' ┆ 120.0    │
│ 1.0    ┆ b'02012011' ┆ b'02'  ┆ b'01' ┆ 120.0    │
│ …      ┆ …           ┆ …      ┆ …     ┆ …        │
│ 72.0   ┆ b'09152011' ┆ b'09'  ┆ b'15' ┆ 110.0    │
│ 72.0   ┆ b'09142011' ┆ b'09'  ┆ b'14' ┆ 110.0    │
│ 72.0   ┆ b'09112011' ┆ b'09'  ┆ b'11' ┆ 110.0    │
│ 72.0   ┆ b'09192011' ┆ b'09'  ┆ b'19' ┆ 110.0    │
│ 72.0   ┆ b'09172011' ┆ b'09'  ┆ b'17' ┆ 110.0    │
└────────┴─────────────┴────────┴───────┴──────────┘


In [6]:
# Cleansing the IMONTH column and Day Column


# Cleaning IMONTH column and renaming to Month


W_YearMonth_2011 = W_Year_2011.rename({'IMONTH': 'Month'})


W_YearMonth_2011 = W_YearMonth_2011.with_columns([
    pl.col('Month').str.replace_all(r"b'", '').str.replace_all('"', '').str.replace_all("'", '')
                    .str.replace_all('01', 'Jan')
                    .str.replace_all('02', 'Feb')
                    .str.replace_all('03', 'Mar')
                    .str.replace_all('04', 'Apr')
                    .str.replace_all('05', 'May')
                    .str.replace_all('06', 'Jun')
                    .str.replace_all('07', 'Jul')
                    .str.replace_all('08', 'Aug')
                    .str.replace_all('09', 'Sep')
                    .str.replace_all('10', 'Oct')
                    .str.replace_all('11', 'Nov')
                    .str.replace_all('12', 'Dec')

])

W_YearMonth_2011


W_YearMonthDay_2011 = W_YearMonth_2011.with_columns(
    pl.col('IDAY').str.replace_all(r"b'", "'").str.replace_all("'", '')
)

W_YearMonthDay_2011 = W_YearMonthDay_2011.with_columns(
    pl.col('IDAY').str.to_integer()
)




shape: (506_467,)
Series: 'DISPCODE' [f64]
[
	120.0
	120.0
	120.0
	120.0
	120.0
	…
	110.0
	110.0
	110.0
	110.0
	110.0
]
